# 01 -- Bronze Layer Ingestion
**Project:** Airbnb Market Analysis -- New York City  
**Layer:** Bronze (raw, unmodified)  
**Source:** Inside Airbnb -- listings.csv.gz  
**Author:** B. Sarang  

This notebook ingests the raw Airbnb listings dataset from the volume into a Bronze Delta table, preserving the source data in its original form.

## 1. Configuration

In [0]:
RAW_PATH = "/Volumes/workspace/default/airbnb_raw/listings.csv.gz"
BRONZE_TABLE = "workspace.default.bronze_airbnb_listings"
BRONZE_PATH = "/Volumes/workspace/default/airbnb_raw/bronze"

## 2. Preview raw file

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(RAW_PATH)
)

print(f"Row count: {df_raw.count()}")
print(f"Column count: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
display(df_raw.limit(10))

## 3. Add ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(RAW_PATH))
)

## 4. Write to Bronze Delta table

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table written: {BRONZE_TABLE}")

## 5. Validate

In [0]:
df_check = spark.table(BRONZE_TABLE)
print(f"Bronze row count: {df_check.count()}")
display(df_check.limit(5))

In [0]:
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").display()

## 6. Transaction log check (Delta time travel)

In [0]:
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").display()